In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

MINIO_ACCESS = "slavakoder"
MINIO_SECRET = "slavakoder"
DB_PASS = "airflow"

spark = SparkSession.builder \
    .appName('cleandata') \
    .config('spark.driver.memory', '2g') \
    .config('spark.executor.memory', '2g') \
    .config('spark.shuffle.partitions', '8') \
    .config("spark.sql.catalog.demo", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.demo.type", "jdbc") \
    .config("spark.sql.catalog.demo.uri", "jdbc:postgresql://postgres:5432/airflow") \
    .config("spark.sql.catalog.demo.jdbc.user", "airflow") \
    .config("spark.sql.catalog.demo.jdbc.password", DB_PASS) \
    .config("spark.sql.catalog.demo.warehouse", "s3a://raw-bronze/warehouse") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS) \
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET) \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.2,org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262,org.postgresql:postgresql:42.6.0") \
    .getOrCreate()
print('запускаемся')

spark.sparkContext.setLogLevel('WARN')

raw_data = "s3a://raw-bronze/landing/p2p_transfers/chunk_1.csv"
dlq_data = 's3a://raw-bronze/dlq/dlq_transfers/'

ddl_schema = "tx_id STRING, sender_id STRING, receiver_id STRING, amount STRING, currency STRING, status STRING, timestamp LONG"


df = spark.read.csv(raw_data, header=True, schema=ddl_schema)

запускаемся


In [2]:
df = (df
    .withColumn('timestamp', F.from_unixtime(F.col('timestamp')).cast('timestamp'))
    .withColumn('sender_id', F.regexp_replace(F.col('sender_id'), r'\s+', ''))
    .withColumn('receiver_id', F.regexp_replace(F.col('receiver_id'), r'\s+', ''))
    .withColumn('amount', F.regexp_replace(F.col('amount'), ',', '.').cast('double'))
    .withColumn('timestamp', F.coalesce(F.col('timestamp'), F.lit('1970-01-01 00:00:00')))
    .withColumn('status', F.coalesce(F.col('status'), F.lit('Unknown')))
)
df = df.fillna('1970-01-01 00:00:00', subset=['timestamp'])
df = df.dropDuplicates(['tx_id',])
df = df.replace(['', 'N/A', 'NULL ', 'NULL'], 'Unknown', subset=['status'])
df = df.sort(F.col('sender_id').asc())
df_kruto = df.filter((F.col("amount") > 0) & (F.col("status").isNotNull()))
df_zalupa = df.filter((F.col("amount") <= 0) | (F.col("status").isNull()))
df_dlq = df_zalupa.withColumn("dlq_processed_at", F.current_timestamp())
df_dlq.write.mode("append").parquet("s3a://raw-bronze/logical_dlq/")

In [3]:
df.show(100, truncate=False)

+--------------------------------+---------+-----------+-------+--------+--------+-------------------+
|tx_id                           |sender_id|receiver_id|amount |currency|status  |timestamp          |
+--------------------------------+---------+-----------+-------+--------+--------+-------------------+
|0e096184060b4e07aba21e2e685cd872|USR_1    |USR_17216  |3626.02|USD     |Unknown |2026-05-17 08:04:07|
|085ad8cef0a141da9860c983620ed8bd|USR_1    |USR_49916  |3044.35|EUR     |SUCCESS |2026-05-19 00:08:30|
|656ddab305dd4b3dbc93183be9467d29|USR_1    |USR_35437  |1157.71|RUB     |Unknown |2026-05-08 05:15:19|
|a5b93561f71f4b88a47e3063c1b50057|USR_1    |USR_12693  |2618.48|RUB     |SUCCESS |2026-05-09 12:07:13|
|404263b3502146e088f4f6ae5da25871|USR_1    |USR_47973  |4949.84|RUB     |PENDING |2026-04-24 13:26:51|
|fd4d0e76d42a41d9bfdf3f315f44b53c|USR_1    |USR_21231  |484.45 |USD     |REJECTED|2026-04-27 00:08:40|
|09964b915f44426084e34075411824e6|USR_1    |USR_48403  |3292.04|KZT     |

In [4]:
df.printSchema()

root
 |-- tx_id: string (nullable = true)
 |-- sender_id: string (nullable = true)
 |-- receiver_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- status: string (nullable = false)
 |-- timestamp: string (nullable = false)

